# Sesi 14: API & Web Scraping dengan Python

Di sesi ini kita belajar dua cara mengambil data dari internet:

1. **API (Application Programming Interface)** - yaitu meminta data dari server secara resmi lewat URL khusus
2. **Web Scraping** - yaitu mengambil data langsung dari halaman HTML website

**Library yang digunakan:**
- 'requests' - untuk HTTP request (sudah include di Anaconda)
- 'BeautifulSoup' - untuk parsing HTML
- 'pandas' - untuk olah hasil data

**Catatan:** Web scraping harus dilakukan secara etis - cek file 'robots.txt'website dan jangan membebani server

In [1]:
# Pastikan BeautifulSoup sudah terinstall
import subprocess
subprocess.run(["pip", "install", "beautifulsoup4", "-q"])
print("Library siap!")

Library siap!


## Bagian 1: Menggunakan API Piblik

Kita akan pakai **Open-Meteo API** - API cuaca gratis, tidak perlu registrasi atau API key
Kita akan ambil data cuaca historis kota Jakarta

In [11]:
import requests
import pandas as pd

# API cuaca Jakarta (koordinat: -6,21, 106.85)
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": -6.21,
    "longitude": 106.85,
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
    "timezone": "Asia/Jakarta",
    "forecast_days": 7
}

response = requests.get(url, params=params)
print(f"Status code: {response.status_code}") # 200 = sukses

data = response.json()
print("Keys tersedia: ", list(data.keys()))

Status code: 200
Keys tersedia:  ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily']


## Ubah hasil API ke DataFrame

Response API berformat JSON - kita ekstrak bagian 'daily' dan jadikan DataFrame

In [13]:
# Ekstrak data harian
daily = data["daily"]

df_cuaca = pd.DataFrame({
    "tanggal":daily["time"],
    "suhu_max": daily["temperature_2m_max"],
    "suhu_min": daily["temperature_2m_min"],
    "curah_hujan_mm": daily["precipitation_sum"]
})

df_cuaca["tanggal"] = pd.to_datetime(df_cuaca["tanggal"])
print(f"Data cuaca Jakarta 7 hari ke depan:")
df_cuaca

Data cuaca Jakarta 7 hari ke depan:


,tanggal,suhu_max,suhu_min,curah_hujan_mm
0,2026-07-01,33.8,24.7,0.0
1,2026-07-02,33.0,26.0,1.7
2,2026-07-03,34.0,25.4,0.0
3,2026-07-04,34.6,26.1,0.0
4,2026-07-05,35.5,26.5,0.0
5,2026-07-06,34.8,26.2,1.5
6,2026-07-07,34.4,26.1,6.6


## Bagian 2: Web Scraping dengan BeautifulSoup

Kita akan scrape tabel dari **Wikipedia** - halaman daftar provinsi di Indonesia beserta jumlah penduduknya
Wikipedia mengizinkan scraping untuk keperluan non-komersial

In [17]:
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/List_of_Indonesian_provinces_by_GDP"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

print(f"Status: {response.status_code}")
print(f"Judul halaman: {soup.title.text}")

Status: 200
Judul halaman: List of Indonesian provinces by GDP - Wikipedia


## Ekstrak Tabel dari HTML

BeautifulSoup memudahkan kita menemukan dan mengambil elemen HTML seperti <table>
Pandas punya fungsi 'read_HTML' yang bisa langsung baca tabel dari HTML string

In [25]:
from io import StringIO

# Cek semua tabel yang ada, cari yang punya kolom GDP
tables = pd.read_html(StringIO(response.text))

for i, t in enumerate(tables):
    print(f"--Tabel {i} | shape: {t.shape} --")
    print(t.columns.tolist())
    print()

--Tabel 0 | shape: (13, 1) --
[0]

--Tabel 1 | shape: (46, 6) --
['Rank', 'Province', 'Region', 'GDP[3] (in billion Rp)', 'GDP Nominal (in billion $)', 'GDP PPP (in billion $)']

--Tabel 2 | shape: (1, 4) --
[0, 1, 2, 3]

--Tabel 3 | shape: (1, 4) --
[0, 1, 2, 3]

--Tabel 4 | shape: (1, 4) --
[0, 1, 2, 3]

--Tabel 5 | shape: (35, 7) --
['Province[16]', '2010[17]', '2011[17]', '2012[17]', '2013[17]', '2014[17]', '2015[17]']

--Tabel 6 | shape: (35, 7) --
['Province[16]', '2016[17]', '2017[17]', '2018[17]', '2019[17]', '2020[17]', '2021[17]']

--Tabel 7 | shape: (39, 3) --
['Province', '2022[17]', '2023']

--Tabel 8 | shape: (2, 2) --
['vteList of Asian subdivisions by GDP', 'vteList of Asian subdivisions by GDP.1']

--Tabel 9 | shape: (9, 2) --
['vteLists of countries by GDP rankings', 'vteLists of countries by GDP rankings.1']



In [33]:
# Tabel 1 adalah data GDP per provinsi
df_gdp = tables[1]

# Bersihkan nama kolom
df_gdp.columns = ["Rank", "Province", "Region", "GDP_Rp_billion", "GDP_USD_billion", "GDP_PPP_billion"]

# Hapus baris yang bukan data (NaN atau header duplikat)
df_gdp = df_gdp.dropna(subset=["Province"])
df_gdp = df_gdp[df_gdp["Province"] != "Province"].reset_index(drop=True)

# Konversi kolom GDP ke numerik
df_gdp["GDP_USD_billion"] = pd.to_numeric(df_gdp["GDP_USD_billion"], errors="coerce")

print(f"Shape: {df_gdp.shape}")
df_gdp.head(10)

Shape: (46, 6)


,Rank,Province,Region,GDP_Rp_billion,GDP_USD_billion,GDP_PPP_billion
0,-,Indonesia,South East Asia,23821104,1445.894,5031.919
1,-,Java Island,Indonesia,13455514,816.723,2842.314
2,-,Sumatra Island,Indonesia,5251355,318.747,1109.285
3,1,Jakarta,Java,3926153,238.310,829.352
4,2,East Java,Java,3403167,206.566,718.878
5,3,West Java,Java,3038668,184.441,641.882
6,4,Central Java,Java,1943188,117.948,410.475
7,-,Kalimantan,Indonesia,1919852,116.531,405.545
8,-,Sulawesi Island,Indonesia,1707775,103.659,360.747
9,5,North Sumatra,Sumatra,1236194,75.035,261.131


In [35]:
# Top 10 provinsi berdasarkan GDP (USD)
top10 = df_gdp.nlargest(10, "GDP_USD_billion")[["Rank", "Province", "Region", "GDP_USD_billion"]]
print("Top 10 Provinsi GDP terbesar (USD Billion):")
top10

Top 10 Provinsi GDP terbesar (USD Billion):


,Rank,Province,Region,GDP_USD_billion
0,-,Indonesia,South East Asia,1445.894
1,-,Java Island,Indonesia,816.723
2,-,Sumatra Island,Indonesia,318.747
3,1,Jakarta,Java,238.310
4,2,East Java,Java,206.566
5,3,West Java,Java,184.441
6,4,Central Java,Java,117.948
7,-,Kalimantan,Indonesia,116.531
8,-,Sulawesi Island,Indonesia,103.659
9,5,North Sumatra,Sumatra,75.035


## Kesimpulan Sesi 14

- **API** >> gunakan 'request.get()' + '.json()' untuk ambil data terstruktur (contoh: Open-Meteo cuaca Jakarta)
- **Web Scraping** >> gunakan 'BeautifulSoup' + 'pd.read_html()' untuk ekstrak tabel dari HTML misalnya Wikipedia
- Selalu cek index tabel yang tepat karena satu halaman bisa punya banyak tabel
- Hasil scraping langsung bisa diolah dengan Panda seperti data CSV biasa
- Data GDP Provinsi Indonesia : **Jawa Barat, DKI Jakarta, Jawa Timur** mendominasi GDP terbesar

**Etika Scraping:** Cek 'robots.txt', gunakan 'User-Agent' header, jangan banjiri server dengan requests

**Next:** Sesi 15 - Dashboard Interaktif dengan Streamlit